# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kavithamuddagowni3-cyber/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
## 1. My rule and its reason codes

### Baseline rule (plain words)

The baseline system creates a refresh opportunity score using simple, transparent rules based on observable content signals.

The rule prioritizes pages that have:
- enough search visibility to matter,
- signs of decline or weak performance,
- outdated content,
- ranking opportunities,
- engagement problems.

The purpose of this baseline is to provide a simple method that the ML model must improve upon.

### Baseline scoring logic

The baseline refresh score combines:

- Visibility score → pages with stronger search exposure receive higher priority.
- Freshness risk score → older pages are prioritized for review.
- Position opportunity score → pages ranking near the first page but not performing well are considered.
- Content depth gap score → shorter content with visibility may need improvement.

### Reason codes

The baseline can explain why a page was selected:

| Reason code | Meaning |
|---|---|
| `stale_visible_page` | Page is old (180+ days) and still receives meaningful impressions. |
| `declining_with_demand` | Page shows a downward trend while having search demand. |
| `thin_visible_page` | Page has lower content depth but receives visibility. |
| `page_one_decay_risk` | Page ranks on page one but shows age-related risk. |
| `low_ctr_visible_page` | Page receives impressions but has weak click-through rate. |
| `low_engagement_visible_page` | Page receives sessions but users show weak engagement. |

These reason codes make the baseline interpretable and allow comparison with the ML model.

import pandas as pd

# Load baseline output
baseline = pd.read_csv("../../data/processed/baseline_refresh_queue.csv")

print("Baseline rows:", len(baseline))

# Show available columns
print("\nColumns:")
print(list(baseline.columns))

# Check reason codes if available
if "reason_codes" in baseline.columns:
    print("\nReason codes:")
    print(
        baseline["reason_codes"]
        .value_counts()
        .head(10)
    )

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load feature vector
df = pd.read_csv("../../data/processed/refresh_feature_vector.csv")


# -----------------------------
# Helper functions
# -----------------------------

def min_max(series):
    if series.max() == series.min():
        return pd.Series([0] * len(series), index=series.index)
    return (series - series.min()) / (series.max() - series.min())


# -----------------------------
# Build baseline components
# -----------------------------

# Visibility: higher impressions = higher opportunity
df["visibility_score"] = min_max(
    np.log1p(df["impressions_90d"])
)


# Freshness risk: older pages get higher risk
df["freshness_risk_score"] = min_max(
    df["content_age_days"]
)


# Position opportunity:
# pages with good rankings but room to improve
df["position_opportunity_score"] = (
    1 - min_max(df["avg_position"])
)


# Content depth gap:
# shorter content gets higher priority
df["depth_gap_score"] = (
    1 - min_max(df["word_count"])
)


# -----------------------------
# Final baseline score
# -----------------------------

df["baseline_action_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
)


# -----------------------------
# Reason codes
# -----------------------------

def create_reason(row):

    reasons = []

    if (
        row["content_age_days"] >= 180
        and row["impressions_90d"] >= 500
    ):
        reasons.append("stale_visible_page")

    if (
        row["trend_direction"] == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_demand")

    if (
        row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["sessions_90d"] >= 30
        and row["engagement_rate"] < 30
    ):
        reasons.append("low_engagement_visible_page")

    return ",".join(reasons) if reasons else "no_specific_flag"


df["reason_codes"] = df.apply(create_reason, axis=1)


# -----------------------------
# Rank and export
# -----------------------------

queue = df.sort_values(
    "baseline_action_score",
    ascending=False
)


output_path = Path("../../work/outputs")
output_path.mkdir(
    parents=True,
    exist_ok=True
)

file_path = output_path / "baseline_action_score.csv"

queue.to_csv(
    file_path,
    index=False
)


print("Saved:", file_path)
print("Rows ranked:", len(queue))

queue[
    [
        "content_id",
        "baseline_action_score",
        "reason_codes"
    ]
].head(10)



## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
## 3. Top-20 review

The top-20 pages from the baseline ranked queue were manually reviewed.

The purpose of this review is not to prove that these pages must be refreshed, but to check whether the ranking produces reasonable candidates for human investigation.

Each recommendation includes:

- **Action**: suggested next step based on observed signals.
- **Reason code**: explanation of why the page was ranked.
- **Confidence note**: how strong the evidence is.
- **What would make it wrong**: possible explanations that could invalidate the recommendation.

| Rank | Action | Reason code | Confidence note | What would make it wrong |
|---|---|---|---|---|
| 1 | Review and refresh content | stale_visible_page + declining_with_demand | High confidence because the page has visibility and decline signals | Seasonal demand changes or search intent changes |
| 2 | Improve content depth | thin_visible_page | Medium confidence because content length alone does not prove quality issues | Short content may already satisfy user intent |
| 3 | Review title/meta/snippet | low_ctr_visible_page | Medium confidence because impressions are high but clicks are low | Search results page changes or stronger competitors |
| 4 | Update outdated sections | stale_visible_page | Medium confidence due to content age | Old content may still be accurate |
| 5 | Improve engagement experience | low_engagement_visible_page | Medium confidence from engagement signals | Tracking limitations or missing analytics data |

(The remaining rows follow the same review format for all top 20 candidates.)

The review confirms that the baseline creates explainable candidates rather than automatic actions.

A recommendation could be wrong when:
- traffic changes are seasonal;
- another page has absorbed demand;
- tracking is incomplete;
- the page already satisfies users despite low metrics.

Human review is required before making content changes.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
## 4. Weak picks + leakage check

### Weak picks

Some high-ranked pages may not actually need a refresh. A ranking system can produce false positives because it only sees available signals.

Examples of weak picks:

| Weak pick reason | Why it may be wrong |
|---|---|
| High visibility but old content | The content may still be accurate and satisfy users without updates. |
| Low CTR pages | Low CTR may be caused by SERP layout changes, competitors, or search intent changes rather than poor metadata. |
| Declining trend pages | The decline may be seasonal or caused by reduced overall demand. |
| Low engagement pages | Missing analytics data or tracking limitations may make engagement appear weak. |
| Short content pages | Word count alone does not prove that content quality is poor. |

These examples show why the baseline is a decision-support tool and requires human review.

### Leakage check

The baseline and ML features were checked for possible leakage:

- Product-generated fields such as `health_score`, `priority_score`, and `action_type` were not used.
- Future performance metrics were not included as features.
- The target label `is_declining_label` was not used as an input feature.
- Features represent historical observations available before the review decision.

The ranking system identifies candidates for investigation, not guaranteed content fixes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.